In [ ]:
# IMPORTS
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import duckdb
import glob
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Match CSV files in the folder
files = glob.glob(r"C:\Users\Galbo\data")

# Take only the first 2
files = files[:2] # I took only two datasets which is from Oct- Nov

# Read and combine
if files:
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f"total rows loaded: {len(df):,}")
    print(f"columns: {list(df.columns)}")
else:
    print("No CSV files found in the folder!")


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# keeping only funnel events cause others are not needed for my study
df = df[df['event_type'].isin(['view', 'cart', 'purchase'])].copy()

# parse datetime
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)

# drop missing critical columns
df = df.dropna(subset=['user_id', 'user_session'])

# one event per session per type (no duplicates)
df = df.drop_duplicates(subset=['user_session', 'event_type'])

# step 5: new columns for analysis
df['date']     = df['event_time'].dt.date
df['hour']     = df['event_time'].dt.hour
df['month']    = df['event_time'].dt.to_period('M').astype(str)
df['dow']      = df['event_time'].dt.day_name()   # day of week
df['price']    = pd.to_numeric(df['price'], errors='coerce').fillna(0)
df['category'] = df['category_code'].fillna('unknown').str.split('.').str[0]

print(f"clean rows: {len(df):,}")
print(f"date range: {df['date'].min()} to {df['date'].max()}")
print(f"unique users: {df['user_id'].nunique():,}")

In [ ]:
# core funnel query — how many sessions reached each stage?

funnel = duckdb.query("""
    WITH
    v AS (SELECT DISTINCT user_session FROM df WHERE event_type = 'view'),
    c AS (SELECT DISTINCT user_session FROM df WHERE event_type = 'cart'),
    p AS (SELECT DISTINCT user_session FROM df WHERE event_type = 'purchase')
    SELECT
        COUNT(DISTINCT v.user_session) AS viewed,
        COUNT(DISTINCT c.user_session) AS carted,
        COUNT(DISTINCT p.user_session) AS purchased
    FROM v
    LEFT JOIN c ON v.user_session = c.user_session
    LEFT JOIN p ON v.user_session = p.user_session""").df().iloc[0]

v2c = round(100 * funnel.carted    / funnel.viewed,  1)
c2p = round(100 * funnel.purchased / funnel.carted,  1)
cvr = round(100 * funnel.purchased / funnel.viewed,  1)

#chart
fig = go.Figure(go.Funnel(
    y = ['Viewed', 'Added to Cart', 'Purchased'],
    x = [int(funnel.viewed), int(funnel.carted), int(funnel.purchased)],
    textinfo = "value+percent previous",
    marker   = dict(color=['#4F46E5', '#7C3AED', '#A855F7'])))
fig.update_layout(title='Overall Purchase Funnel', height=380)
fig.show()

print(f"  Out of {int(funnel.viewed):,} sessions that viewed a product:")
print(f"  + Only {v2c}% added to cart")
print(f"  + Of those, only {c2p}% actually purchased")
print(f"  + Overall, just {cvr}% of all sessions result in a purchase")
print(f"  + That means {round(100-cvr,1)}% of potential buyers are being lost somewhere")

In [ ]:
# waterfall chart — shows absolute users lost at each stage
# makes it very clear where the biggest problem is

dropped_at_cart     = int(funnel.viewed  - funnel.carted)
dropped_at_checkout = int(funnel.carted  - funnel.purchased)

fig = go.Figure(go.Waterfall(
    orientation = 'v',
    measure = ['absolute', 'relative', 'relative', 'total'],
    x = ['Started', 'Lost at Cart', 'Lost at Checkout', 'Purchased'],
    y = [int(funnel.viewed), -dropped_at_cart, -dropped_at_checkout, 0],
    text = [f"{int(funnel.viewed):,}", f"-{dropped_at_cart:,}",f"-{dropped_at_checkout:,}", f"{int(funnel.purchased):,}"],textposition = 'outside',
    increasing = dict(marker=dict(color='#10B981')),
    decreasing = dict(marker=dict(color='#EF4444')),
    totals= dict(marker=dict(color='#6366F1'))))
fig.update_layout(title='User Drop-off at Each Stage', height=420, showlegend=False)
fig.show()

print(f"  {dropped_at_cart:,} users ({round(100-v2c,1)}%) left WITHOUT adding anything to cart")
print(f"  {dropped_at_checkout:,} users ({round(100-c2p,1)}%) added to cart but NEVER bought")

## Biggest problem is at the BOTTOM of the funnel

In [ ]:
# understanding the volume of each event type across the whole dataset
# before deduplication — use df_raw or recount from df

event_counts = df['event_type'].value_counts().reset_index()
event_counts.columns = ['event_type', 'count']

fig = px.pie(event_counts, values='count', names='event_type',title='Share of Each Event Type',color_discrete_sequence=['#4F46E5', '#F59E0B', '#10B981'],hole=0.4)
fig.update_layout(height=380)
fig.show()

total = event_counts['count'].sum()
for _, row in event_counts.iterrows():
    pct = round(100 * row['count'] / total, 1)
    print(f"  {row['event_type']:10} : {row['count']:,}  ({pct}% of all events)")

In [ ]:
# pivot trick: one row per session with 1/0 flags for each stage
# then group by category to see conversion per category

cat_df = duckdb.query("""
    WITH p AS (
        SELECT user_session,
               MAX(category) AS category,
               MAX(CASE WHEN event_type='view'     THEN 1 ELSE 0 END) AS viewed,
               MAX(CASE WHEN event_type='cart'     THEN 1 ELSE 0 END) AS carted,
               MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
        FROM df
        GROUP BY user_session
    )
    SELECT
        category,
        SUM(viewed)    AS sessions,
        SUM(carted)    AS carted,
        SUM(purchased) AS purchased,
        ROUND(100.0 * SUM(carted)    / NULLIF(SUM(viewed),0), 1) AS v2c_pct,
        ROUND(100.0 * SUM(purchased) / NULLIF(SUM(carted),0),  1) AS c2p_pct,
        ROUND(100.0 * SUM(purchased) / NULLIF(SUM(viewed),0),  1) AS cvr_pct
    FROM p
    WHERE category != 'unknown'
    GROUP BY category
    HAVING SUM(viewed) >= 100
    ORDER BY sessions DESC
""").df()

# grouped bar: v2c and c2p side by side per category for ease
fig = go.Figure()
fig.add_trace(go.Bar(name='View→Cart %',    x=cat_df['category'], y=cat_df['v2c_pct'], marker_color='#6366F1'))
fig.add_trace(go.Bar(name='Cart→Purchase %',x=cat_df['category'], y=cat_df['c2p_pct'], marker_color='#F59E0B'))
fig.update_layout(barmode='group', title='Funnel Conversion by Category',height=420, xaxis_tickangle=-30,legend=dict(orientation='h', y=1.1))
fig.show()

best  = cat_df.loc[cat_df['cvr_pct'].idxmax()]
worst = cat_df.loc[cat_df['cvr_pct'].idxmin()]
print(f"  Best  CVR: '{best['category']}' at {best['cvr_pct']}%")
print(f"  Worst CVR: '{worst['category']}' at {worst['cvr_pct']}%")
print(cat_df[['category','sessions','v2c_pct','c2p_pct','cvr_pct']].to_string(index=False))

In [ ]:
brand_df = duckdb.query("""
    WITH p AS (
        SELECT user_session,
               MAX(brand) AS brand,
               MAX(CASE WHEN event_type='view'     THEN 1 ELSE 0 END) AS viewed,
               MAX(CASE WHEN event_type='cart'     THEN 1 ELSE 0 END) AS carted,
               MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
        FROM df WHERE brand IS NOT NULL
        GROUP BY user_session
    )
    SELECT brand,
           SUM(viewed)    AS sessions,
           SUM(carted)    AS carted,
           SUM(purchased) AS purchased,
           ROUND(100.0 * SUM(carted)    / NULLIF(SUM(viewed),0), 1) AS v2c_pct,
           ROUND(100.0 * SUM(purchased) / NULLIF(SUM(carted),0),  1) AS c2p_pct,
           ROUND(100.0 * SUM(purchased) / NULLIF(SUM(viewed),0),  1) AS cvr_pct
    FROM p
    GROUP BY brand
    HAVING SUM(viewed) >= 300
    ORDER BY sessions DESC
    LIMIT 15
""").df()

fig = px.bar(brand_df.sort_values('cvr_pct', ascending=True),
             x='cvr_pct', y='brand', orientation='h',
             title='Overall CVR % — Top 15 Brands by Traffic',
             text='cvr_pct', color='cvr_pct',
             color_continuous_scale='Purples')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, coloraxis_showscale=False)
fig.show()


best_b  = brand_df.loc[brand_df['cvr_pct'].idxmax()]
worst_b = brand_df.loc[brand_df['cvr_pct'].idxmin()]
print(f"  Best  brand CVR: {best_b['brand']} at {best_b['cvr_pct']}%")
print(f"  Worst brand CVR: {worst_b['brand']} at {worst_b['cvr_pct']}%")
# check if high views = low trust
high_view_low_cvr = brand_df.nlargest(3,'sessions')[['brand','sessions','cvr_pct']]
print(f"\n  Top 3 most-viewed brands and their CVRs:")
print(high_view_low_cvr.to_string(index=False))

In [ ]:
# to fing which day and hour has peak perfomance

# hour of day
hourly = duckdb.query("""
    WITH p AS (
        SELECT EXTRACT(HOUR FROM ANY_VALUE(event_time)) AS hour, 
               user_session,
               MAX(CASE WHEN event_type='view'     THEN 1 ELSE 0 END) AS viewed,
               MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
        FROM df 
        GROUP BY hour, user_session
    )
    SELECT hour,
           SUM(viewed)    AS sessions,
           SUM(purchased) AS purchases,
           ROUND(100.0 * SUM(purchased) / NULLIF(SUM(viewed),0), 2) AS cvr_pct
    FROM p GROUP BY hour ORDER BY hour
""").df()

fig = px.bar(hourly, x='hour', y='cvr_pct',
             color='cvr_pct', color_continuous_scale='Purples',
             title='CVR % by Hour of Day (0=midnight, 12=noon)',
             text=hourly['cvr_pct'].map(lambda x: f'{x}%'))
fig.update_traces(textposition='outside')
fig.update_layout(height=370, coloraxis_showscale=False)
fig.show()

peak_hour = int(hourly.loc[hourly['cvr_pct'].idxmax(), 'hour'])
low_hour  = int(hourly.loc[hourly['cvr_pct'].idxmin(), 'hour'])

# day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow = duckdb.query("""
    WITH p AS (
        SELECT dow, user_session,
               MAX(CASE WHEN event_type='view'     THEN 1 ELSE 0 END) AS viewed,
               MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
        FROM df GROUP BY dow, user_session
    )
    SELECT dow,
           SUM(viewed)    AS sessions,
           SUM(purchased) AS purchases,
           ROUND(100.0 * SUM(purchased) / NULLIF(SUM(viewed),0), 2) AS cvr_pct
    FROM p GROUP BY dow
""").df()

dow['dow'] = pd.Categorical(dow['dow'], categories=dow_order, ordered=True)
dow = dow.sort_values('dow')

fig = px.bar(dow, x='dow', y='cvr_pct', color='cvr_pct',
             color_continuous_scale='Blues',
             title='CVR % by Day of Week',
             text=dow['cvr_pct'].map(lambda x: f'{x}%'))
fig.update_traces(textposition='outside')
fig.update_layout(height=370, coloraxis_showscale=False)
fig.show()

best_day  = dow.loc[dow['cvr_pct'].idxmax(), 'dow']
worst_day = dow.loc[dow['cvr_pct'].idxmin(), 'dow']

print(f"  Best  hour to convert: {peak_hour:02d}:00  (highest purchase intent)")
print(f"  Worst hour to convert: {low_hour:02d}:00  (avoid scheduling campaigns here)")
print(f"  Best  day of week: {best_day}")
print(f"  Worst day of week: {worst_day}")
print(f"  + Schedule email campaigns at {peak_hour:02d}:00 on {best_day} for maximum CVR")

In [ ]:
# total lost revenue
rev = duckdb.query("""
    WITH
    carted    AS (SELECT DISTINCT user_session FROM df WHERE event_type='cart'),
    purchased AS (SELECT DISTINCT user_session FROM df WHERE event_type='purchase'),
    abandoned AS (
        SELECT df.user_session, df.price, df.category
        FROM df
        JOIN carted c ON df.user_session = c.user_session
        WHERE df.event_type = 'cart'
          AND df.user_session NOT IN (SELECT user_session FROM purchased)
    )
    SELECT
        COUNT(DISTINCT user_session)     AS abandoned_carts,
        ROUND(SUM(price), 2)            AS total_lost,
        ROUND(SUM(price)*0.10, 2)       AS recovery_10pct,
        ROUND(AVG(price), 2)            AS avg_cart_value
    FROM abandoned
""").df().iloc[0]

print("=" * 50)
print("    ABANDONED CART REVENUE SUMMARY")
print("=" * 50)
print(f"  Abandoned sessions : {int(rev.abandoned_carts):,}")
print(f"  Total revenue lost : ${float(rev.total_lost):,.2f}")
print(f"  Average cart value : ${float(rev.avg_cart_value):,.2f}")
print(f"  10% recovery =     : ${float(rev.recovery_10pct):,.2f}")
print("=" * 50)

# abandoned revenue broken down by category
cat_rev = duckdb.query("""
    WITH
    carted    AS (SELECT DISTINCT user_session FROM df WHERE event_type='cart'),
    purchased AS (SELECT DISTINCT user_session FROM df WHERE event_type='purchase'),
    abandoned AS (
        SELECT df.category, df.price
        FROM df
        JOIN carted c ON df.user_session = c.user_session
        WHERE df.event_type = 'cart'
          AND df.user_session NOT IN (SELECT user_session FROM purchased)
    )
    SELECT category,
           COUNT(*)            AS abandoned_count,
           ROUND(SUM(price),2) AS revenue_lost
    FROM abandoned
    WHERE category != 'unknown'
    GROUP BY category
    ORDER BY revenue_lost DESC
""").df()

fig = px.bar(cat_rev, x='category', y='revenue_lost',
             title='Revenue Lost to Abandoned Carts by Category',
             text=cat_rev['revenue_lost'].map(lambda x: f'${x:,.0f}'),
             color='revenue_lost', color_continuous_scale='Reds')
fig.update_traces(textposition='outside')
fig.update_layout(height=400, coloraxis_showscale=False)
fig.show()

In [ ]:
# bucket products by price range and see if expensive = harder to sell

price_df = duckdb.query("""
    WITH p AS (
        SELECT user_session,
               MAX(price) AS price,
               MAX(CASE WHEN event_type='view'     THEN 1 ELSE 0 END) AS viewed,
               MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
        FROM df GROUP BY user_session
    )
    SELECT
        CASE
            WHEN price = 0        THEN '1. Free / Unknown'
            WHEN price < 10       THEN '2. Under $10'
            WHEN price < 30       THEN '3. $10 - $30'
            WHEN price < 100      THEN '4. $30 - $100'
            ELSE                       '5. Over $100'
        END AS price_bucket,
        COUNT(*)       AS sessions,
        SUM(purchased) AS purchased,
        ROUND(100.0 * SUM(purchased) / NULLIF(COUNT(*),0), 1) AS cvr_pct
    FROM p
    WHERE viewed = 1
    GROUP BY price_bucket
    ORDER BY price_bucket
""").df()

fig = px.bar(price_df, x='price_bucket', y='cvr_pct',title='CVR % by Price Range — Do Higher Prices Convert Less?',text='cvr_pct', color='cvr_pct',color_continuous_scale='RdYlGn')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=400, coloraxis_showscale=False,xaxis_title='Price Bucket', yaxis_title='CVR %')
fig.show()


print(price_df[['price_bucket','sessions','cvr_pct']].to_string(index=False))
high = price_df[price_df['price_bucket']=='5. Over $100']['cvr_pct'].values
low  = price_df[price_df['price_bucket']=='3. $10 - $30']['cvr_pct'].values
if len(high) and len(low):
    if high[0] < low[0]:
        print(f"\n  + Expensive products (${'>'}100) convert less ({high[0]}%) than mid-price ({low[0]}%)")
    else:
        print("\n  + Higher price does NOT reduce CVR here — customers trust the brand")

In [ ]:
# how many users bought more than once?
# to fingd loyal customers

user_df = duckdb.query("""
    SELECT user_id,
           COUNT(DISTINCT user_session) AS sessions,
           SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchases,
           ROUND(SUM(price), 2) AS total_spent
    FROM df
    GROUP BY user_id""").df()

user_df['buyer_type'] = user_df['purchases'].apply(lambda x: 'Non-buyer' if x == 0 else ('One-time' if x == 1 else 'Repeat buyer'))
summary = user_df.groupby('buyer_type').agg(
    users       = ('user_id','count'),
    avg_sessions= ('sessions','mean'),
    avg_spent   = ('total_spent','mean')).round(2).reset_index()

fig = px.pie(summary, values='users', names='buyer_type',title='User Split: Non-buyer vs One-time vs Repeat Buyer',color_discrete_sequence=['#EF4444','#6366F1','#10B981'],hole=0.4)
fig.update_layout(height=380)
fig.show()

print(summary.to_string(index=False))
repeat = summary[summary['buyer_type']=='Repeat buyer']
onetime = summary[summary['buyer_type']=='One-time']
if not repeat.empty and not onetime.empty:
    spend_lift = round(float(repeat['avg_spent'].values[0]) / float(onetime['avg_spent'].values[0]), 1)
    print(f"\n  Repeat buyers spend {spend_lift}x more than one-time buyers on average")
    print("  + Focus retention efforts: loyalty rewards, personalised emails etc..")

In [ ]:
# these products attract attention but fail to convert

gap_df = duckdb.query("""
    WITH p AS (
        SELECT product_id,
               SUM(CASE WHEN event_type='view'     THEN 1 ELSE 0 END) AS views,
               SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchases
        FROM df
        GROUP BY product_id
    )
    SELECT product_id,
           views, purchases,
           ROUND(100.0 * purchases / NULLIF(views,0), 2) AS cvr_pct,
           views - purchases AS gap
    FROM p
    WHERE views >= 50
    ORDER BY views DESC
    LIMIT 20
""").df()

fig = px.scatter(gap_df, x='views', y='purchases',
                 size='gap', color='cvr_pct',
                 color_continuous_scale='RdYlGn',
                 title='Views vs Purchases per Product (size = gap, color = CVR%)',
                 hover_data=['product_id','cvr_pct'],
                 labels={'views':'Total Views','purchases':'Total Purchases'})
fig.update_layout(height=430)
fig.show()


print("Products with HIGH views but LOW conversion:")
worst_products = gap_df.nsmallest(5, 'cvr_pct')[['product_id','views','purchases','cvr_pct']]
print(worst_products.to_string(index=False))